## 1. What is DummyClassifier?

DummyClassifier is a simple baseline classifier from scikit-learn.

It makes predictions using a basic rule and ignores the relationships between the input features and the target. It may inspect the target labels
during `.fit()`, but it does not learn patterns such as relationships between age, balance, job, and subscription outcome.

```python
from sklearn.dummy import DummyClassifier
```

The purpose of a dummy classifier is to answer:

> Does the real model perform better than a very simple prediction strategy?

A real classifier should generally provide meaningful improvements over this baseline.

`DummyClassifier` supports:

- Binary classification, such as `"yes"` and `"no"`
- Multiclass classification, such as `"low"`, `"medium"`, and `"high"`
- Some multi-output classification problems

It should not be treated as the final predictive model.

## 2. Basic DummyClassifier workflow

```python
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)
```

The process is:

```text
Create the dummy classifier
            ↓
Fit it using the training target
            ↓
Generate simple predictions for the test set
            ↓
Compare predictions with the actual test outcomes
```

Although `.fit()` receives `X_train`, dummy strategies ignore the feature values.

## 3. `DummyClassifier` strategies

### `strategy="most_frequent"`

Always predicts the most common class found in the training target.

```python
dummy_model = DummyClassifier(strategy="most_frequent")
```

If 88% of the training outcomes are `"no"`, every prediction will be `"no"`.

This is useful for showing why accuracy can be misleading on an imbalanced dataset.

---

### `strategy="prior"`

Predicts the most frequent class, like "most_frequent".

```python
dummy_model = DummyClassifier(strategy="prior")
```

The difference concerns predicted probabilities:

- `"most_frequent"` assigns probability 1 to the most frequent class.
- `"prior"` returns the observed training-class proportions.

For example, `"prior"` might return probabilities near:

```text
P(no)  = 0.88
P(yes) = 0.12
```

`"prior"` is the default strategy in current scikit-learn versions.

---

### `strategy="stratified"`

Randomly generates predictions according to the class proportions found in the training target.

```python
dummy_model = DummyClassifier(
    strategy="stratified",
    random_state=RANDOM_SEED
)
```

If the training target is 88% `"no"` and 12% `"yes"`, predictions are randomly generated with approximately those probabilities.

Because this strategy is random, `random_state` should be set for reproducibility.

---

### `strategy="uniform"`

Randomly predicts every class with equal probability.

```python
dummy_model = DummyClassifier(
    strategy="uniform",
    random_state=RANDOM_SEED
)
```

For a binary target:

```text
P(no)  = 0.50
P(yes) = 0.50
```

It ignores the observed class proportions.

---

### `strategy="constant"`

Always predicts a class specified by the user.

```python
dummy_model = DummyClassifier(
    strategy="constant",
    constant="yes"
)
```

This can be useful for examining what happens when every observation is classified as the minority class.

## 4. Important DummyClassifier methods

### `.fit()`

Fits the baseline using the training data.

```python
dummy_model.fit(X_train, y_train)
```

Depending on its strategy, the classifier may learn simple information from `y_train`, such as:

- Which class is most common
- The proportion belonging to each class
- Which class labels exist

It does not learn predictive relationships from the features.

---

### `.predict()`

Returns a predicted class for each observation.

```python
y_pred = dummy_model.predict(X_test)
```

Example output:

```text
["no", "no", "no", "no"]
```

---

### `.predict_proba()`

Returns the predicted probability of each class.

```python
y_probability = dummy_model.predict_proba(X_test)
```

The order of the probability columns can be inspected with:

```python
dummy_model.classes_
```

If the output is:

```python
array(["no", "yes"])
```

then:

`y_probability[:, 0]`

contains the `"no"` probabilities, while:

`y_probability[:, 1]`

contains the `"yes"` probabilities.

---

### `.score()`

For a classifier, `.score()` normally returns mean accuracy.

```python
dummy_model.score(X_test, y_test)
```

This is equivalent to:

```python
accuracy_score(y_test, y_pred)
```

Accuracy alone may be misleading for an imbalanced dataset.


# Classification terminology

## 5. Positive and negative classes

For the bank-marketing project:

```text
Positive class = "yes"
Negative class = "no"
```

“Positive” does not necessarily mean good. It means the event the model is intended to detect.

Here, the positive event is:

> The customer subscribes to a term deposit.

## 6. The four possible prediction outcomes

### True positive (`TP`)

The actual outcome is `"yes"` and the prediction is `"yes"`.

> The model correctly identifies a subscriber.

### True negative (`TN`)

The actual outcome is `"no"` and the prediction is `"no"`.

> The model correctly identifies a non-subscriber.

### False positive (`FP`)

The actual outcome is `"no"`, but the prediction is `"yes"`.

> The model predicts that a customer will subscribe, but the customer does not.

This is also called a Type I error.

### False negative (`FN`)

The actual outcome is `"yes"`, but the prediction is `"no"`.

> The model misses a customer who actually subscribes.

This is also called a Type II error.

# Confusion matrix

## 7. What is a confusion matrix?

A confusion matrix counts the four possible prediction outcomes.

```python
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=["no", "yes"]
)

print(matrix)
```

With the label order `["no", "yes"]`, scikit-learn produces:

```python
[[TN, FP],
  [FN, TP]]
```

The rows represent actual classes, and the columns represent predicted classes:

```text
                          Predicted
                      no             yes
Actual no            TN             FP
Actual yes           FN             TP
```

For example:

```text
[[7985,    0],
  [1058,    0]]
```

means:

- 7,985 true negatives
- 0 false positives
- 1,058 false negatives
- 0 true positives

The model predicted "no" for every observation.

The individual values can be extracted with:

```python
tn, fp, fn, tp = matrix.ravel()
```

# Metrics based on predicted classes

## 8. Accuracy

Accuracy measures the proportion of all predictions that were correct.

```text
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```

```python
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
```

Accuracy answers:

> Out of all observations, what proportion did the model classify correctly?

Its values range from 0 to 1:

- 1.0 means every prediction was correct.
- 0.0 means no predictions were correct.

---

### Limitation

Accuracy can be misleading when classes are imbalanced.

If 88% of customers have the outcome "no", a model that always predicts "no" receives approximately 88% accuracy while detecting no subscribers.

## 9. Balanced accuracy

Balanced accuracy gives equal importance to the positive and negative classes.

For binary classification:

```text
Balanced accuracy = (Recall + Specificity) / 2
```

```python
from sklearn.metrics import balanced_accuracy_score

balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
```

Balanced accuracy answers:

> How well does the model identify both classes when each class is given equal importance?

A classifier that always predicts the majority class will usually have balanced accuracy near `0.5` in a binary problem, even if ordinary accuracy is high.

Balanced accuracy is especially useful for imbalanced targets.

## 10. Precision

Precision measures how many predicted positives are actually positive.

```text
Precision = TP / (TP + FP)
```

```python
from sklearn.metrics import precision_score

precision = precision_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)
```

Precision answers:

> Of all customers predicted to subscribe, what proportion actually subscribed?

High precision means the model produces relatively few false-positive predictions.

Precision becomes important when acting on a positive prediction is costly.

For example, if the bank can contact only a small number of customers, it may want positive predictions to be reliable.

---

### Limitation

A model can achieve high precision by predicting `"yes"` only in a few extremely certain cases. It could therefore have high precision but low recall.

## 11. Recall

Recall measures how many actual positives the model successfully identifies.

```text
Recall = TP / (TP + FN)
```

```python
from sklearn.metrics import recall_score

recall = recall_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)
```

Recall is also called:

- Sensitivity
- True-positive rate (`TPR`)

Recall answers:

> Of all customers who actually subscribed, what proportion did the model identify?

High recall means the model produces relatively few false negatives.

Recall becomes important when missing a positive case is costly.

---

### Limitation

A model could obtain perfect recall by predicting `"yes"` for everyone. However, this would probably create many false positives and have poor precision.

## 12. Specificity

Specificity measures how many actual negatives the model correctly identifies.

```text
Specificity = TN / (TN + FP)
```

Specificity is also called the true-negative rate (`TNR`).

It answers:

> Of all customers who did not subscribe, what proportion did the model correctly identify as non-subscribers?

It can be calculated using the confusion matrix:

```text
specificity = tn / (tn + fp)
```

It can also be calculated by treating "no" as the class of interest:

```python
specificity = recall_score(
    y_test,
    y_pred,
    pos_label="no"
)
```

A model that predicts `"no"` for everyone has perfect specificity but zero recall for `"yes"`.

## 13. False-positive rate

The false-positive rate measures the proportion of actual negatives incorrectly classified as positive.

```text
False-positive rate = FP / (FP + TN)
```

It can also be expressed as:

```text
False-positive rate = 1 - Specificity

false_positive_rate = fp / (fp + tn)
```

It answers:

> Of all customers who did not subscribe, what proportion did the model incorrectly classify as subscribers?

## 14. False-negative rate

The false-negative rate measures the proportion of actual positives missed by the model.

```text
False-negative rate = FN / (FN + TP)
```

It can also be expressed as:

```text
False-negative rate = 1 - Recall

false_negative_rate = fn / (fn + tp)
```

It answers:

> Of all customers who subscribed, what proportion did the model miss?

## 15. F1 score

The F1 score combines precision and recall using their harmonic mean.

```text
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

It can also be calculated directly from the confusion-matrix values:

```text
F1 = 2TP / (2TP + FP + FN)
```

```python
from sklearn.metrics import f1_score

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)
```

F1 answers:

> How well does the model balance precision and recall for the positive class?

Its values range from 0 to 1:

- 1.0 is best.
- 0.0 is worst.

F1 is useful for imbalanced classification when both false positives and false negatives matter.

---

### Limitation

F1 ignores true negatives. It may not fully represent performance when correctly identifying the negative class is important.

## 16. F-beta score

The F-beta score is a variation of F1 that allows precision or recall to receive more importance.

```python
from sklearn.metrics import fbeta_score
```

### `beta = 1`

Gives precision and recall equal importance and is equivalent to F1.

```python
f1 = fbeta_score(
    y_test,
    y_pred,
    beta=1,
    pos_label="yes"
)
```

### `beta > 1`

Gives recall more importance.

```python
f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2,
    pos_label="yes"
)
```

F2 is useful when missing positive cases is particularly costly.

### `beta < 1`

Gives precision more importance.

```python
f_half = fbeta_score(
    y_test,
    y_pred,
    beta=0.5,
    pos_label="yes"
)
```

This is useful when false-positive predictions are particularly costly.

## 17. Negative predictive value

Negative predictive value measures how many predicted negatives are actually negative.

```text
Negative predictive value = TN / (TN + FN)

negative_predictive_value = tn / (tn + fn)
```

It answers:

> Of all customers predicted not to subscribe, what proportion actually did not subscribe?

It can be viewed as the negative-class counterpart to positive-class precision.

## 18. Matthews correlation coefficient

The Matthews correlation coefficient (`MCC`) combines all four confusion-matrix outcomes.

```python
from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(y_test, y_pred)
```

Its usual range is from `-1` to `1`:

- `1` means perfect predictions.
- `0` means performance no better than random association.
- `-1` means complete disagreement between predictions and actual outcomes.

MCC can be informative for imbalanced classification because it considers:

- True positives
- True negatives
- False positives
- False negatives

## 19. Cohen’s kappa

Cohen’s kappa measures agreement between actual and predicted classes while accounting for agreement that could occur by chance.

```python
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(y_test, y_pred)
```

Its typical interpretation is:

- `1` means perfect agreement.
- `0` means agreement similar to chance.
- Values below `0` indicate worse-than-chance agreement.

Kappa is helpful when high accuracy might partly result from dominant classes.

# Metrics based on predicted probabilities

## 20. Predicted classes versus predicted probabilities

`.predict()` returns final class labels:

```python
y_pred = model.predict(X_test)
```

Example:

```python
["no", "yes", "no"]
```

`.predict_proba()` returns probabilities for each class:

```python
y_proba = model.predict_proba(X_test)
```

To obtain the probability of `"yes"` safely:

```python
positive_class_index = list(model.classes_).index("yes")
y_score = y_proba[:, positive_class_index]
```

Example:

```text
[0.08, 0.76, 0.31]
```

Metrics such as accuracy, precision, recall, and F1 normally use final class predictions.

Metrics such as ROC AUC, average precision, log loss, and Brier score use probability estimates or decision scores.

## 21. Classification threshold

A classification threshold converts a probability into a predicted class.

A common default threshold is `0.5`:

```text
Probability of yes >= 0.5 → predict yes
Probability of yes < 0.5  → predict no
```

Example:

```python
y_pred_threshold = np.where(y_score >= 0.5, "yes", "no")
```

Lowering the threshold usually:

- Increases recall
- Increases the number of positive predictions
- May reduce precision
- May increase false positives

Raising the threshold usually:

- Increases the required confidence for a positive prediction
- May increase precision
- Usually reduces recall
- May increase false negatives

The threshold must be chosen using training or validation data, not the final test set.

## 22. ROC curve

The receiver operating characteristic (`ROC`) curve measures performance across many classification thresholds.

```python
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_test,
    y_score,
    pos_label="yes"
)
```

The ROC curve plots:

```text
y-axis: True-positive rate (recall)
x-axis: False-positive rate
```

It shows the tradeoff between detecting positives and incorrectly flagging negatives.

## 23. ROC AUC

ROC AUC is the area under the ROC curve.

```python
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(
    y_test,
    y_score
)
```

If the target contains strings, scikit-learn normally determines the class ordering. To make the positive class completely explicit, it can first be
converted:

```python
y_test_binary = (y_test == "yes").astype(int)

roc_auc = roc_auc_score(
    y_test_binary,
    y_score
)
```

General interpretation:

- `1.0`: perfect ranking
- `0.5`: approximately random ranking
- `Below 0.5`: rankings are generally reversed

ROC AUC answers:

> How well does the model tend to rank positive observations above negative observations across all thresholds?

---

### Limitation

ROC AUC may appear optimistic when the positive class is rare because the false-positive rate is divided by a large number of negatives.

## 24. Precision-recall curve

The precision-recall curve shows the tradeoff between precision and recall across different thresholds.

```python
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(
    y_test,
    y_score,
    pos_label="yes"
)
```

It plots:

```text
y-axis: Precision
x-axis: Recall
```

It is especially useful when:

- The positive class is rare
- Performance on the positive class is most important
- The negative class greatly outnumbers the positive class

## 25. Average precision and PR AUC

Average precision (`AP`) summarizes the precision-recall curve.

```python
from sklearn.metrics import average_precision_score

average_precision = average_precision_score(
    y_test,
    y_score,
    pos_label="yes"
)
```

It rewards models that maintain high precision while achieving greater recall.

For an uninformative classifier, average precision is typically near the proportion of positive observations. Therefore, if approximately 12% of cases are positive, a value near `0.12` provides useful baseline context.

“PR AUC” is sometimes used loosely to describe average precision, but they are not always calculated identically. When using `average_precision_score`,
report the result specifically as average precision.

## 26. Log loss

Log loss evaluates the quality of predicted probabilities.

```python
from sklearn.metrics import log_loss

loss = log_loss(
    y_test,
    model.predict_proba(X_test),
    labels=model.classes_
)
```

Lower values are better.

Log loss heavily penalizes predictions that are confidently wrong.

For example:

- Predicting `0.51` for `"yes"` when the outcome is `"no"` receives a moderate penalty.
- Predicting `0.999` for `"yes"` when the outcome is `"no"` receives a much larger penalty.

Unlike accuracy, log loss distinguishes between uncertain and highly confident predictions.

## 27. Brier score

The Brier score measures the mean squared difference between predicted positive-class probabilities and actual binary outcomes.

```python
from sklearn.metrics import brier_score_loss

y_test_binary = (y_test == "yes").astype(int)

brier = brier_score_loss(
    y_test_binary,
    y_score
)
```

Lower values are better:

- `0` means perfect probabilities.
- Larger values mean the probabilities are less accurate.

The Brier score evaluates probability quality rather than only final class predictions.

It reflects both:

- Discrimination: separating positives from negatives
- Calibration: whether predicted probabilities match observed frequencies

# Reporting several metrics

## 28. classification_report

classification_report displays precision, recall, F1, and support for each class.

```python
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        labels=["no", "yes"],
        zero_division=0
    )
)
```

Example structure:

```text
              precision    recall  f1-score   support

          no       ...
         yes       ...

    accuracy       ...
   macro avg       ...
weighted avg       ...
```

---

### Support

Support is the number of actual observations belonging to a class.

For example:

```text
support for no  = number of actual "no" outcomes
support for yes = number of actual "yes" outcomes
```

Support is a count, not a performance score.

## 29. Binary averaging

For binary classification, metrics such as precision, recall, and F1 can focus on one positive class.

```python
precision_score(
    y_test,
    y_pred,
    pos_label="yes",
    average="binary",
    zero_division=0
)
```

`average="binary"` is the usual default for binary targets.

It calculates the score for the class identified by `pos_label`.

## 30. Macro average

Macro averaging calculates the metric separately for every class and then takes the unweighted mean.

```python
Macro average = (score for no + score for yes) / 2

f1_macro = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)
```

Each class receives equal importance, regardless of its size.

Macro averaging is useful when minority-class performance should matter as much as majority-class performance.

## 31. Weighted average

Weighted averaging calculates the metric for every class and weights each result by that class’s support.

```python
f1_weighted = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)
```

A large majority class has more influence over the result.

A weighted score can therefore look strong even when performance on the minority class is weak.

## 32. Micro average

Micro averaging combines the TP, FP, and FN counts across all classes before calculating the metric.

```python
f1_micro = f1_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)
```

In ordinary single-label multiclass classification, micro precision, micro recall, micro F1, and accuracy are generally equal.

Micro averaging gives each observation equal importance rather than giving each class equal importance.

## 33. zero_division

Some metrics become mathematically undefined when a model never predicts a particular class.

For example, positive-class precision is:

```text
Precision = TP / (TP + FP)
```

If the model never predicts "yes", then:

```text
TP + FP = 0
```

This would require division by zero.

The argument:

```text
zero_division=0
```

tells scikit-learn to return 0 instead of producing an undefined result warning.

```python
precision_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)
```

This does not improve the model. It only controls how the undefined metric is reported.

# Evaluation code

## 34. Common evaluation metrics for the bank-marketing project

```python
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)
```

# Final class predictions
```python
y_pred = model.predict(X_test)
```

# Positive-class probabilities
```python
positive_class_index = list(model.classes_).index("yes")
y_score = model.predict_proba(X_test)[:, positive_class_index]
```

# Binary version of the actual target for probability metrics
```python
y_test_binary = (y_test == "yes").astype(int)
```

# Confusion-matrix counts
```python
matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=["no", "yes"]
)

tn, fp, fn, tp = matrix.ravel()
```

# Class-prediction metrics
```python
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="yes",
    zero_division=0
)

f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2,
    pos_label="yes",
    zero_division=0
)

specificity = tn / (tn + fp)
false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)
negative_predictive_value = tn / (tn + fn)

mcc = matthews_corrcoef(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)
```

# Probability and ranking metrics
```python
roc_auc = roc_auc_score(y_test_binary, y_score)
average_precision = average_precision_score(
    y_test_binary,
    y_score
)

loss = log_loss(
    y_test,
    model.predict_proba(X_test),
    labels=model.classes_
)

brier = brier_score_loss(
    y_test_binary,
    y_score
)

print("Accuracy:", accuracy)
print("Balanced accuracy:", balanced_accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("Specificity:", specificity)
print("F1:", f1)
print("F2:", f2)
print("False-positive rate:", false_positive_rate)
print("False-negative rate:", false_negative_rate)
print("Negative predictive value:", negative_predictive_value)
print("Matthews correlation coefficient:", mcc)
print("Cohen's kappa:", kappa)
print("ROC AUC:", roc_auc)
print("Average precision:", average_precision)
print("Log loss:", loss)
print("Brier score:", brier)

print("\nConfusion matrix:")
print(matrix)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["no", "yes"],
        zero_division=0
    )
)
```

Not every project requires every metric. For this bank-marketing checkpoint, the required metrics are:

- Accuracy
- Precision
- Recall
- F1
- Confusion matrix

Balanced accuracy and average precision are especially useful additional metrics because the target is imbalanced.

## 35. How to choose relevant metrics

The best metric depends on the practical cost of different mistakes.

### When false positives are expensive

Prioritize:

- Precision
- Specificity
- False-positive rate

Example:

> The bank has limited staff and does not want to spend resources contacting customers who are unlikely to subscribe.

### When false negatives are expensive

Prioritize:

- Recall
- False-negative rate
- F2 score

Example:

> The bank does not want to miss customers who would subscribe.

### When both mistakes matter

Consider:

- F1
- Balanced accuracy
- Matthews correlation coefficient
- Confusion matrix

### When ranking customers matters

Consider:

- ROC AUC
- Average precision
- Precision-recall curve

Example:

> The bank wants to rank customers and contact the most promising ones first.

### When probability quality matters

Consider:

- Log loss
- Brier score
- Calibration analysis

Example:

> The bank needs the predicted probability itself to represent a meaningful estimate of subscription likelihood.

## 36. Interpreting the dummy baseline

Suppose the dummy baseline produces:

```text
Accuracy:  0.883
Precision: 0.000
Recall:    0.000
F1:        0.000

Confusion matrix:
[[7985,    0],
  [1058,    0]]
```

This means the classifier predicted "no" for every observation.

Its accuracy is high only because "no" is the majority class. It failed to identify any actual subscribers.

A suitable interpretation is:

> * The most-frequent dummy classifier achieved approximately 88.3% accuracy by predicting “no” for every observation. However, its precision, recall, and
> * F1 score for the positive class were all zero because it identified none of the 1,058 subscribers.
> * This demonstrates that accuracy is not meaningful by itself for this imbalanced target. The dummy result provides a baseline against which the trained classifier should be compared.

Documentation:

- DummyClassifier (https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html),
- classification metrics (https://scikit-learn.org/stable/modules/model_evaluation.html),
- and confusion matrix (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).